# Trajectories of magnetic soft robot

### Imports

In [1]:
import numpy as np
import jax.numpy as jnp
import jax
import optax
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import softmobility as sm

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

In [2]:
BLUEACCENT="#3274B5" 
REDACCENT="#DA3B26"
GREY="#929292"

## Simulation of default parameters

### YAML file

In [3]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - spring_k
  - radius
input_names:
  - gravity
  - active_force
    
# Default Values (Optional)
defaults:
  spring_k: .5           
  radius: .33
  _spr_: 50
  _rad_: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: 1
    position: [0, 0, 1]                    
    orientation: [phi, 0, 0]
    force: [-gravity0, -gravity1, -gravity2]
    torque: [-_spr_ * spring_k * phi, 0, 0]              

  - # Sphere 1 #################
    radius: _rad_ * radius               
    position: [0, 0, -radius]
    orientation: [-phi/_rad_/radius, 0, 0]
    force: [gravity0, gravity1 + active_force * sin(phi/_rad_/radius), gravity2 + active_force * cos(phi/_rad_/radius)]
    torque: [_spr_ * spring_k * phi, 0, 0]
"""

In [4]:
yaml_data_rigid = """
# Variable Prefixes (Optional)
design_names:      
  - radius
input_names:
  - gravity
  - active_force
    
# Default Values (Optional)
defaults:
  radius: .5
  _rad_: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: 1
    position: [0, 0, 1]                    
    orientation: [0, 0, 0]
    force: [-gravity0, -gravity1, -gravity2]
    torque: [0, 0, 0]              

  - # Sphere 1 #################
    radius: _rad_ * radius               
    position: [0, 0, -radius]
    orientation: [0, 0, 0]
    force: [gravity0, gravity1, gravity2 + active_force]
    torque: [0, 0, 0]
"""

### Soft Body

In [5]:
mybody= sm.SoftBody(yaml_data, verbose=True)
myrigidbody = sm.SoftBody(yaml_data_rigid, verbose=False)

  Found variables: phi, radius, spring_k, gravity0, gravity1, gravity2, active_force
  3D field inputs:  ['gravity']
  Scalar inputs:    ['active_force']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '1']
      Orientation: ['phi', '0', '0']
      Coupling matrix C_H:
[['-1', '0', '0', '0'], ['0', '-1', '0', '0'], ['0', '0', '-1', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['-50*spring_k'], ['0'], ['0']]
    Sphere 1
      Radius: radius
      Position: ['0', '0', '-radius']
      Orientation: ['-phi/radius', '0', '0']
      Coupling matrix C_H:
[['1', '0', '0', '0'], ['0', '1', '0', 'sin(phi/radius)'], ['0', '0', '1', 'cos(phi/radius)'], ['0', '0', '0', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['50*spring_k'], ['0'], ['0']]


### Parameters of flow and gravity field

In [6]:
# Calculating force to have a swimming velocity of 1
tensors = myrigidbody.compute_fast_mobility(dofs=jnp.zeros(myrigidbody.Ndof))
FORCE_INTENSITY=1/tensors.M_H[2,-1]

In [7]:
# Create a TGV flow with with max vorticy = 1
FLOW = sm.Taylor_Green_flow(omega=2)
# Create a gravity field 
GRAVITY = sm.gravity_field(g=50.0)
# Create an active force scalar
FORCE = sm.Scalar(lambda pos, t, param: param[0], param_shape=(1,))
FORCE.update_from_parameter(FORCE_INTENSITY)

In [8]:
# parameters of time integration
FINAL_TIME = 4 * jnp.pi  
DT = 0.1
N_TRAJECTORIES = 15

Y_INIT_POSITIONS = np.linspace(np.pi/N_TRAJECTORIES/2, 2*np.pi, N_TRAJECTORIES, endpoint=False)
N_DT = int(FINAL_TIME / DT)

In [9]:
ROLLOUT = sm.FlowBodyRollout(
    soft_body = mybody,
    flow      = FLOW,
    input_map = {"gravity": GRAVITY, "active_force": FORCE},
)

### Objective and optimization

In [10]:
def my_objective(rollout, design):
    dofs0        = jnp.ones(mybody.Ndof) * 1e-6   # jnp.zeros causes a singularity
    tensors    = mybody.compute_mobility_problem(dofs0, design)
    force_norm = 1.0 / tensors.M_H[2, 3]
    FORCE.update_from_parameter(force_norm)
    
    # One rollout per initial position, average Z velocity
    velocities = jnp.stack([
        rollout.rollout(
            dt          = DT,
            n_steps     = N_DT,
            init_position = jnp.array([jnp.pi/2, x_init, 0.0]),
            init_dofs   = dofs0,
            design      = design,
        )[0][-1, 2] / FINAL_TIME
        for x_init in Y_INIT_POSITIONS
    ])
    return jnp.mean(velocities)

In [ ]:
opt = sm.FlowBodyOptimizer(ROLLOUT, my_objective, optimizer=optax.sgd(0.01))

optimal_design = opt.run(
    init_design = 0.5 * jnp.ones(mybody.Ndesign),
    n_steps     = 1000,
    print_every = 100,
    clip_min    = 0.01,
    clip_max    = 1.0,
    grad_clip   = 0.1,
    maximize    = True,
)

step    0  objective=0.8912  |grad|=0.1000  design0=0.50  design1=0.50
step  100  objective=1.0189  |grad|=0.1000  design0=0.43  design1=0.42
step  200  objective=1.1324  |grad|=0.1000  design0=0.38  design1=0.34
step  300  objective=1.2126  |grad|=0.0086  design0=0.32  design1=0.29
step  400  objective=1.2126  |grad|=0.0026  design0=0.31  design1=0.29
step  500  objective=1.2126  |grad|=0.0012  design0=0.31  design1=0.29


### Surfer functions

In [ ]:
tensors = mybody.compute_fast_mobility(dofs=jnp.zeros(mybody.Ndof), design=optimal_design)

TAU_ALIGN = 0.5/tensors.M_H[3,1]/50
print(TAU_ALIGN)

In [ ]:
from jax.scipy.linalg import expm

E_Z = jnp.array([0.0, 0.0, 1.0])   # reference "upward" direction

def _rollout_surfer(tau, x_init):
    """Surfer agent: swims at speed 1, aligns with z @ exp(tau * grad_u)."""

    position = jnp.array([np.pi/2, x_init, 0.0])
    hatp     = E_Z.copy()   # initial swimming direction: upward

    def step(carry, t):
        position, hatp = carry
        time = t * DT

        u_lab              = FLOW.velocity(position, time)
        omega_lab, _       = FLOW.omega_rate_of_strain(position, time)
        grad_u             = FLOW.gradient(position, time)   # 3×3 matrix

        # --- Preferred direction ---
        n    = E_Z @ expm(tau * grad_u)         # deform current heading
        hatn = n / jnp.linalg.norm(n)           # normalize

        # --- hatp ODE (on the unit sphere) ---
        #   d hatp/dt = omega × hatp + (hatn - (hatn·hatp) hatp) / (2 talign)
        vorticity_term  = jnp.cross(omega_lab, hatp)
        alignment_term  = (hatn - jnp.dot(hatn, hatp) * hatp) / (2.0 * TAU_ALIGN)
        dhatp           = vorticity_term + alignment_term

        # --- Position ODE: advected by flow + swimming at speed 1 ---
        dpos = u_lab + hatp

        # --- RK2 ---
        pos_mid  = position + DT * dpos  / 2
        hatp_mid = hatp     + DT * dhatp / 2
        hatp_mid = hatp_mid / jnp.linalg.norm(hatp_mid)   # keep on sphere

        u_mid            = FLOW.velocity(pos_mid, time + DT/2)
        omega_mid, _     = FLOW.omega_rate_of_strain(pos_mid, time + DT/2)
        grad_u_mid       = FLOW.gradient(pos_mid, time + DT/2)

        n_mid    = E_Z @ expm(tau * grad_u_mid)
        hatn_mid = n_mid / jnp.linalg.norm(n_mid)

        dhatp_mid = (jnp.cross(omega_mid, hatp_mid)
                     + (hatn_mid - jnp.dot(hatn_mid, hatp_mid) * hatp_mid) / (2.0 * TAU_ALIGN))
        dpos_mid  = u_mid + hatp_mid

        position = position + DT * (dpos + dpos_mid) / 2
        hatp     = hatp     + DT * (dhatp + dhatp_mid) / 2
        hatp     = hatp     / jnp.linalg.norm(hatp)   # project back to S²

        return (position, hatp), position   # save position for trajectory

    (position, hatp), positions = jax.lax.scan(
        step, (position, hatp), jnp.arange(N_DT)
    )

    return position[2] / FINAL_TIME, positions   # mean Z velocity + full path


def mean_velocity_surfer(tau):
    velocities  = jnp.stack([_rollout_surfer(tau, x)[0] for x in Y_INIT_POSITIONS])
    return -jnp.mean(velocities)

grad_fn_surfer = jax.jit(jax.value_and_grad(mean_velocity_surfer))

### Surfer optimization

In [ ]:
initial_guess = 0.5 * jnp.ones(1)

optimal_surfer, optimal_velocity = gradient_descent(
    grad_fn_surfer,
    initial_guess,
    lr          = 0.01,
    n_steps     = 1000,
    print_every = 100,
    clip_min=0.01,
    clip_max=2.0,
)

### Simulations for plots

In [ ]:
DT = 0.1
N_DT = int(FINAL_TIME / DT)

tensors = mybody.compute_fast_mobility(dofs=jnp.zeros(mybody.Ndof), design=optimal_design)
FORCE_INTENSITY = 1.0/tensors.M_H[2,-1]
FORCE = constant_scalar(value=FORCE_INTENSITY)

In [ ]:
surfer_trajectories = []

for yinit in Y_INIT_POSITIONS:
    v, p = _rollout_surfer(tau=optimal_surfer, x_init=yinit)
    print("New initial position:", yinit, ", velocity:", v)
    surfer_trajectories.append(p[:, np.newaxis, :])
    
end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in surfer_trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")

In [ ]:
mybody.set_design_defaults(new_design=optimal_design)

trajectories = []

for x_init in Y_INIT_POSITIONS:
    print("New initial position", x_init)
    solver = FlowBodySolver(
        soft_body=mybody, 
        flow=FLOW, 
        input_map={"gravity": GRAVITY, "active_force": FORCE}, 
        dt=DT, 
        init_position=[np.pi/2, x_init, 0],
        init_orientation=[0, 0, 0], 
        integrator="RK2")
    trajectories.append(solver.simulate(final_time=FINAL_TIME))
    
end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")

In [ ]:
myrigidbody.set_design_defaults(new_design=[optimal_design[0]])

rigid_trajectories = []

for x_init in Y_INIT_POSITIONS:
    print("New initial position", x_init)
    solver = FlowBodySolver(
        soft_body=myrigidbody, 
        flow=FLOW, 
        input_map={"gravity": GRAVITY, "active_force": FORCE}, 
        dt=DT, 
        init_position=[np.pi/2, x_init, 0],
        init_orientation=[0, 0, 0], 
        integrator="RK2")
    rigid_trajectories.append(solver.simulate(final_time=FINAL_TIME))
    
end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in rigid_trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")

### Preparing the figure of trajectory

In [ ]:
# --- Helper to build tick labels like "−2π", "−π/2", "0", "π", etc. ---
def pi_ticks(lo, hi, step_pi=1):
    """Return (tickvals, ticktext) with spacing of step_pi * π."""
    import math
    n_lo = math.floor(lo / (step_pi * np.pi))
    n_hi = math.ceil(hi  / (step_pi * np.pi))
    vals, texts = [], []
    for n in range(n_lo, n_hi + 1):
        v = n * step_pi * np.pi
        vals.append(v)
        k = n * step_pi          # multiple of π
        if   k == 0:  texts.append('0')
        elif k == 1:  texts.append('π')
        elif k == -1: texts.append('−π')
        else:         texts.append(f'{k}π')
    return vals, texts

def bounds_2pi(lo, hi, step_pi=1):
    """Round lo down and hi up to the nearest multiple of step_pi*π."""
    unit = step_pi * np.pi
    return np.floor(lo / unit) * unit, np.ceil(hi / unit) * unit

In [ ]:
colors=[BLUEACCENT, REDACCENT, GREY]

# --- Compute global bounds across ALL trajectories ---
all_y = np.concatenate([np.array(jnp.array([t[0] for t in traj])[:, 1]) for traj in trajectories + rigid_trajectories + surfer_trajectories])
all_z = np.concatenate([np.array(jnp.array([t[0] for t in traj])[:, 2]) for traj in trajectories + rigid_trajectories + surfer_trajectories])

Y_MIN, Y_MAX = bounds_2pi(all_y.min(), all_y.max())
Z_MIN, Z_MAX = bounds_2pi(all_z.min(), all_z.max())

GRID_STEP = 1
y_ticks, _ = pi_ticks(Y_MIN, Y_MAX, step_pi=GRID_STEP)
z_ticks, _ = pi_ticks(Z_MIN, Z_MAX, step_pi=GRID_STEP)

ASPECT     = (Y_MAX - Y_MIN)
FIG_WIDTH  = 1000   # total width for both panels
FIG_HEIGHT = int((FIG_WIDTH / 3) * (Z_MAX - Z_MIN) / ASPECT)

fig = make_subplots(rows=1, cols=3, subplot_titles=("rigid", "soft", "surfer"),
                    horizontal_spacing=0.05)

#
ny, nz = 300, 300
y_grid = np.linspace(Y_MIN, Y_MAX, ny)
z_grid = np.linspace(Z_MIN, Z_MAX, nz)
Y, Z   = np.meshgrid(y_grid, z_grid)

omega = 1.0
PSI   = 0.5 * omega * np.sin(Y) * np.sin(Z)

# --- Streamlines: add to all subplots ---
for col in [1, 2, 3]:
    fig.add_trace(go.Contour(
        x=y_grid, y=z_grid, z=PSI,
        ncontours = 8,
        contours  = dict(coloring='none', showlines=True),
        line      = dict(color=GREY, width=0.8),
        showscale = False,
        showlegend= False,
        hoverinfo = 'skip',
    ), row=1, col=col)

# --- Helper to add one set of trajectories to a given col ---
def add_trajectories(fig, traj_list, col):
    for (trajectory, x_init) in zip(traj_list, Y_INIT_POSITIONS):
        positions = jnp.array([t[0] for t in trajectory])
        y_pos = np.array(positions[:, 1])
        z_pos = np.array(positions[:, 2])
        label = f"y₀={x_init/np.pi:.2g}π"

        fig.add_trace(go.Scatter(
            x=y_pos, y=z_pos,
            mode='lines',
            line=dict(color=colors[col-1], width=2),
            name=label, showlegend=False,
            hovertemplate=f"Y=%{{x:.3f}}<br>Z=%{{y:.3f}}<extra>{label}</extra>",
        ), row=1, col=col)
        
        fig.add_trace(go.Scatter(
            x=[y_pos[-1]], y=[z_pos[-1]],
            mode='markers',
            marker=dict(color=colors[col-1], size=8, symbol=['circle', 'square']),
            showlegend=False,
            hovertemplate=f"%{{text}}<extra>{label}</extra>",
            text=['start', 'end'],
        ), row=1, col=col)

add_trajectories(fig, rigid_trajectories,  col=1)
add_trajectories(fig, trajectories,        col=2)
add_trajectories(fig, surfer_trajectories, col=3)

# --- Shared axis style ---
axis_style = dict(
    showticklabels=False,
    showgrid=True, gridcolor=GREY, gridwidth=1,
    zeroline=True, linecolor=GREY, mirror=True,
    title=None,
    constrain='domain',
)

fig.update_layout(
    template='plotly_white',
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    margin=dict(l=20, r=20, t=30, b=20),
    showlegend=False,
    xaxis=dict(**axis_style, range=[Y_MIN, Y_MAX], scaleanchor='y',  scaleratio=1, tickvals=y_ticks),
    yaxis=dict(**axis_style, range=[Z_MIN, Z_MAX], constraintoward='center', tickvals=z_ticks),
    xaxis2=dict(**axis_style, range=[Y_MIN, Y_MAX], scaleanchor='y2', scaleratio=1, tickvals=y_ticks),
    yaxis2=dict(**axis_style, range=[Z_MIN, Z_MAX], constraintoward='center', tickvals=z_ticks),
    xaxis3=dict(**axis_style, range=[Y_MIN, Y_MAX], scaleanchor='y3', scaleratio=1, tickvals=y_ticks),
    yaxis3=dict(**axis_style, range=[Z_MIN, Z_MAX], constraintoward='center', tickvals=z_ticks),
    )


### Figure

In [ ]:
fig.show()